In [1]:
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings.ollama import OllamaEmbeddings
from neo4j_graphrag.llm.ollama_llm import OllamaLLM
from neo4j_graphrag.retrievers import VectorRetriever
from neo4j_graphrag.generation import GraphRAG

In [2]:
driver = GraphDatabase.driver(
    "bolt://localhost:7687/",
    auth=("neo4j","ava25-DB!!")
)

In [3]:
embedder = OllamaEmbeddings(model="mistral-embed")
llm = OllamaLLM(model_name="llama3")
retriever = VectorRetriever(driver=driver, embedder=embedder)
graphrag = GraphRAG(driver=driver, retriever=retriever, llm=llm)


TypeError: VectorRetriever.__init__() missing 1 required positional argument: 'index_name'

In [ ]:
graph = Neo4jGraph(
    url="bolt://localhost:7687/",
    username="neo4j",
    password="ava25-DB!!"
)
graph.refresh_schema()
print(graph.schema)

Node properties:
Entity {sub_type: STRING, type: STRING, label: STRING, y: FLOAT, id: STRING, name: STRING, x: FLOAT}
Event {sub_type: STRING, type: STRING, label: STRING, y: FLOAT, id: STRING, timestamp: STRING, x: FLOAT, monitoring_type: STRING, findings: STRING, content: STRING, results: STRING, assessment_type: STRING, destination: STRING, movement_type: STRING, outcome: STRING, enforcement_type: STRING, participants: INTEGER, activity_type: STRING, reference: STRING, date: STRING, time: STRING}
Commodity {type: STRING, label: STRING, sub_type: STRING, x: FLOAT, y: FLOAT, id: STRING, name: STRING}
Relationship {sub_type: STRING, type: STRING, label: STRING, y: FLOAT, id: STRING, friendship_type: STRING, x: FLOAT, permission_type: STRING, start_date: STRING, end_date: STRING, report_type: STRING, submission_date: STRING, jurisdiction_type: STRING, authority_level: STRING, coordination_type: STRING, operational_role: STRING}
Relationship properties:
RELATED_TO {is_inferred: BOOLEAN, 

In [ ]:
llm = OllamaLLM(model="gemma3:1b")

In [ ]:
template_string = """
You are an expert in knowledge graphs. You answer questions about people, objects, and events using Cypher queries for Neo4j.

Only return the Cypher query. Use valid Cypher syntax (commas instead of semicolons in property maps).

Here is the graph schema:
{schema}

Examples:
Question: Who is Angela Merkel?
Cypher:
MATCH (e:Entity {{sub_type: 'Person', name: 'Angela Merkel'}}) RETURN e

Question: What happened on May 3rd?
Cypher:
MATCH (e:Event {{date: '2023-05-03'}}) RETURN e

Question: What did Nadja Conti do?
Cypher:
MATCH (e:Entity {{name: 'Nadja Conti'}})-[:RELATED_TO]->(ev:Event)  
RETURN ev

Question: What commodities are related to Angela Merkel?
Cypher:
MATCH (e:Entity {{name: 'Angela Merkel'}})-[:RELATED_TO]->(c:Commodity)  
RETURN c

Question: Who participated in events on May 3rd?
Cypher:
MATCH (ev:Event {{date: '2023-05-03'}})<-[:RELATED_TO]-(e:Entity)  
RETURN e

Question: Who sent an event to Nadja Conti?
Cypher:
MATCH (sender:Entity)-[:sent]->(ev:Event)-[:received]->(receiver:Entity {{name: 'Nadja Conti'}})  
RETURN sender


Here is the question you should build the cypher query for:

Question: {question}
Cypher:
"""

cypher_prompt = PromptTemplate(
    input_variables=["question", "schema"],
    template=template_string
)


cypher_generator = LLMChain(llm=llm, prompt=cypher_prompt)

In [ ]:
def extract_cypher_query(response_text: str) -> str:
    # Optional: Entfernt alles vor einem eventuellen ```cypher-Block
    match = re.search(r"```cypher\n(.*?)```", response_text, re.DOTALL)
    if match:
        return match.group(1).strip()
    
    # Fallback: Suche nach "MATCH" und gib alles danach zurück
    match = re.search(r"(MATCH\s*\(.*)", response_text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    
    return response_text.strip()

def remove_think_blocks(text: str) -> str:
    """
    Entfernt alle <think>...</think> Blöcke aus dem gegebenen Text.
    """
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

In [ ]:
def query_graph(question: str):
    cypher = cypher_generator.invoke({
        "question": question,
        "schema": graph.schema
    })
    print(cypher)
    cypher_raw = cypher["text"]
    cypher_clean = remove_think_blocks(cypher_raw)
    cypher_clean = extract_cypher_query(cypher_clean)
    print(cypher_clean)

    print("Generated Cypher:\n", cypher_clean)
    try:
        result = graph.query(cypher_clean)
        return result
    except Exception as e:
        print("Fehler beim Ausführen von Cypher:", e)
        return None


In [ ]:
frage = "Who are different Persons there?"
antwort = query_graph(question=frage)
print(antwort)

{'question': 'Who are different Persons there?', 'schema': 'Node properties:\nEntity {sub_type: STRING, type: STRING, label: STRING, y: FLOAT, id: STRING, name: STRING, x: FLOAT}\nEvent {sub_type: STRING, type: STRING, label: STRING, y: FLOAT, id: STRING, timestamp: STRING, x: FLOAT, monitoring_type: STRING, findings: STRING, content: STRING, results: STRING, assessment_type: STRING, destination: STRING, movement_type: STRING, outcome: STRING, enforcement_type: STRING, participants: INTEGER, activity_type: STRING, reference: STRING, date: STRING, time: STRING}\nCommodity {type: STRING, label: STRING, sub_type: STRING, x: FLOAT, y: FLOAT, id: STRING, name: STRING}\nRelationship {sub_type: STRING, type: STRING, label: STRING, y: FLOAT, id: STRING, friendship_type: STRING, x: FLOAT, permission_type: STRING, start_date: STRING, end_date: STRING, report_type: STRING, submission_date: STRING, jurisdiction_type: STRING, authority_level: STRING, coordination_type: STRING, operational_role: STR

In [4]:
import requests

class OllamaLLM:
    def __init__(self, model="gemma3:1b", base_url="http://localhost:11434"):
        self.model = model
        self.base_url = base_url

    def invoke(self, prompt: str):
        response = requests.post(
            f"{self.base_url}/api/generate",
            json={"model": self.model, "prompt": prompt, "stream": False},
        )
        response.raise_for_status()
        result_text = response.json()["response"].strip()
        return type("LLMResponse", (), {"content": result_text})()

In [5]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver("bolt://localhost:7687/", auth=("neo4j", "ava25-DB!!"))

llm = OllamaLLM() 
examples = [
    {
        "question": "Who are the persons in the database?",
        "cypher": "MATCH (e:Entity {sub_type: 'Person'}) RETURN DISTINCT e.name"
    },
    {
        "question": "Which events is Alice involved in?",
        "cypher": "MATCH (e:Entity {name: 'Alice'})-[:RELATED_TO]->(ev:Event) RETURN DISTINCT ev"
    },
    {
        "question": "What commodities are related to John?",
        "cypher": "MATCH (e:Entity {name: 'John'})-[:RELATED_TO]->(c:Commodity) RETURN DISTINCT c.name"
    },
    {
        "question": "Which entities received events?",
        "cypher": "MATCH (ev:Event)-[:received]->(e:Entity) RETURN DISTINCT e.name"
    },
    {
        "question": "What type of relationships exist between entities?",
        "cypher": "MATCH (e1:Entity)-[r:RELATED_TO]->(rel:Relationship)-[:RELATED_TO]->(e2:Entity) RETURN DISTINCT r.type"
    },
    {
        "question": "Which events were sent by Bob?",
        "cypher": "MATCH (e:Entity {name: 'Bob'})-[:sent]->(ev:Event) RETURN DISTINCT ev"
    },
    {
        "question": "What is the timestamp of the events related to Alice?",
        "cypher": "MATCH (e:Entity {name: 'Alice'})-[:RELATED_TO]->(ev:Event) RETURN DISTINCT ev.timestamp"
    },
    {
        "question": "What entities are connected to the event 'Inspection_2023'?",
        "cypher": "MATCH (ev:Event {label: 'Inspection_2023'})-[:RELATED_TO]->(e:Entity) RETURN DISTINCT e.name"
    },
    {
        "question": "Which commodities are linked to any event?",
        "cypher": "MATCH (ev:Event)-[:evidence_for]->(c:Commodity) RETURN DISTINCT c.name"
    },
    {
        "question": "What outcomes are recorded in events involving Charlie?",
        "cypher": "MATCH (e:Entity {name: 'Charlie'})-[:RELATED_TO]->(ev:Event) RETURN DISTINCT ev.outcome"
    }
]



schema = graph.schema
retriever = Text2CypherRetriever(
    driver=driver,
    llm=llm,
    neo4j_schema=schema,
    examples=examples
)

NameError: name 'graph' is not defined

In [6]:
query_text = "which persons are talking to the boss"
print(retriever.search(query_text=query_text))


NameError: name 'retriever' is not defined

items=[RetrieverResultItem(content="<Record n.name='Sam'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sam'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sam'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sam'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sam'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sam'>", metadata=None), RetrieverResultItem(content="<Record n.name='Kelly'>", metadata=None), RetrieverResultItem(content="<Record n.name='Kelly'>", metadata=None), RetrieverResultItem(content="<Record n.name='Nadia Conti'>", metadata=None), RetrieverResultItem(content="<Record n.name='Nadia Conti'>", metadata=None), RetrieverResultItem(content="<Record n.name='Nadia Conti'>", metadata=None), RetrieverResultItem(content="<Record n.name='Nadia Conti'>", metadata=None), RetrieverResultItem(content="<Record n.name='Nadia Conti'>", metadata=None), RetrieverResultItem(content="<Record n.name='Nadia Conti'>", metadata=None), RetrieverResultItem(content="<Record n.name='Nadia Conti'>", metadata=None), RetrieverResultItem(content="<Record n.name='Nadia Conti'>", metadata=None), RetrieverResultItem(content="<Record n.name='Nadia Conti'>", metadata=None), RetrieverResultItem(content="<Record n.name='Elise'>", metadata=None), RetrieverResultItem(content="<Record n.name='Elise'>", metadata=None), RetrieverResultItem(content="<Record n.name='Elise'>", metadata=None), RetrieverResultItem(content="<Record n.name='Elise'>", metadata=None), RetrieverResultItem(content="<Record n.name='Elise'>", metadata=None), RetrieverResultItem(content="<Record n.name='Liam Thorne'>", metadata=None), RetrieverResultItem(content="<Record n.name='Liam Thorne'>", metadata=None), RetrieverResultItem(content="<Record n.name='Liam Thorne'>", metadata=None), RetrieverResultItem(content="<Record n.name='Liam Thorne'>", metadata=None), RetrieverResultItem(content="<Record n.name='Liam Thorne'>", metadata=None), RetrieverResultItem(content="<Record n.name='Liam Thorne'>", metadata=None), RetrieverResultItem(content="<Record n.name='Liam Thorne'>", metadata=None), RetrieverResultItem(content="<Record n.name='Liam Thorne'>", metadata=None), RetrieverResultItem(content="<Record n.name='Liam Thorne'>", metadata=None), RetrieverResultItem(content="<Record n.name='Liam Thorne'>", metadata=None), RetrieverResultItem(content="<Record n.name='Liam Thorne'>", metadata=None), RetrieverResultItem(content="<Record n.name='Liam Thorne'>", metadata=None), RetrieverResultItem(content="<Record n.name='Liam Thorne'>", metadata=None), RetrieverResultItem(content="<Record n.name='Liam Thorne'>", metadata=None), RetrieverResultItem(content="<Record n.name='Liam Thorne'>", metadata=None), RetrieverResultItem(content="<Record n.name='Samantha Blake'>", metadata=None), RetrieverResultItem(content="<Record n.name='Samantha Blake'>", metadata=None), RetrieverResultItem(content="<Record n.name='Samantha Blake'>", metadata=None), RetrieverResultItem(content="<Record n.name='Samantha Blake'>", metadata=None), RetrieverResultItem(content="<Record n.name='Samantha Blake'>", metadata=None), RetrieverResultItem(content="<Record n.name='Samantha Blake'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Davis'>", metadata=None), RetrieverResultItem(content="<Record n.name='Rodriguez'>", metadata=None), RetrieverResultItem(content="<Record n.name='Rodriguez'>", metadata=None), RetrieverResultItem(content="<Record n.name='Rodriguez'>", metadata=None), RetrieverResultItem(content="<Record n.name='Rodriguez'>", metadata=None), RetrieverResultItem(content="<Record n.name='Rodriguez'>", metadata=None), RetrieverResultItem(content="<Record n.name='Rodriguez'>", metadata=None), RetrieverResultItem(content="<Record n.name='Rodriguez'>", metadata=None), RetrieverResultItem(content="<Record n.name='Rodriguez'>", metadata=None), RetrieverResultItem(content="<Record n.name='Rodriguez'>", metadata=None), RetrieverResultItem(content="<Record n.name='Rodriguez'>", metadata=None), RetrieverResultItem(content="<Record n.name='Rodriguez'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sailor Shift'>", metadata=None), RetrieverResultItem(content="<Record n.name='Clepper Jensen'>", metadata=None), RetrieverResultItem(content="<Record n.name='Clepper Jensen'>", metadata=None), RetrieverResultItem(content="<Record n.name='Clepper Jensen'>", metadata=None), RetrieverResultItem(content="<Record n.name='Clepper Jensen'>", metadata=None), RetrieverResultItem(content="<Record n.name='Clepper Jensen'>", metadata=None), RetrieverResultItem(content="<Record n.name='Clepper Jensen'>", metadata=None), RetrieverResultItem(content="<Record n.name='Clepper Jensen'>", metadata=None), RetrieverResultItem(content="<Record n.name='Miranda Jordan'>", metadata=None), RetrieverResultItem(content="<Record n.name='Miranda Jordan'>", metadata=None), RetrieverResultItem(content="<Record n.name='Miranda Jordan'>", metadata=None), RetrieverResultItem(content="<Record n.name='Miranda Jordan'>", metadata=None), RetrieverResultItem(content="<Record n.name='Miranda Jordan'>", metadata=None), RetrieverResultItem(content="<Record n.name='Miranda Jordan'>", metadata=None), RetrieverResultItem(content="<Record n.name='Miranda Jordan'>", metadata=None), RetrieverResultItem(content="<Record n.name='Miranda Jordan'>", metadata=None), RetrieverResultItem(content="<Record n.name='Miranda Jordan'>", metadata=None), RetrieverResultItem(content="<Record n.name='Miranda Jordan'>", metadata=None), RetrieverResultItem(content="<Record n.name='Miranda Jordan'>", metadata=None), RetrieverResultItem(content="<Record n.name='Miranda Jordan'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Intern'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Intern'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Intern'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Intern'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Intern'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Intern'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Intern'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Intern'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Intern'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Intern'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Intern'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Intern'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Intern'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Lookout'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Lookout'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Lookout'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Lookout'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Lookout'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Lookout'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Lookout'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Lookout'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Lookout'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Lookout'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Lookout'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Accountant'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Accountant'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Accountant'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Accountant'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Accountant'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mrs. Money'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mrs. Money'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mrs. Money'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mrs. Money'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mrs. Money'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mrs. Money'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mrs. Money'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mrs. Money'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mrs. Money'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mrs. Money'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mrs. Money'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mrs. Money'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Middleman'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Middleman'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Middleman'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Middleman'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Middleman'>", metadata=None), RetrieverResultItem(content="<Record n.name='The Middleman'>", metadata=None), RetrieverResultItem(content="<Record n.name='Boss'>", metadata=None), RetrieverResultItem(content="<Record n.name='Boss'>", metadata=None), RetrieverResultItem(content="<Record n.name='Boss'>", metadata=None), RetrieverResultItem(content="<Record n.name='Boss'>", metadata=None), RetrieverResultItem(content="<Record n.name='Boss'>", metadata=None), RetrieverResultItem(content="<Record n.name='Boss'>", metadata=None), RetrieverResultItem(content="<Record n.name='Boss'>", metadata=None), RetrieverResultItem(content="<Record n.name='Boss'>", metadata=None), RetrieverResultItem(content="<Record n.name='Boss'>", metadata=None), RetrieverResultItem(content="<Record n.name='Boss'>", metadata=None), RetrieverResultItem(content="<Record n.name='Boss'>", metadata=None), RetrieverResultItem(content="<Record n.name='Boss'>", metadata=None), RetrieverResultItem(content="<Record n.name='Small Fry'>", metadata=None), RetrieverResultItem(content="<Record n.name='Small Fry'>", metadata=None), RetrieverResultItem(content="<Record n.name='Small Fry'>", metadata=None), RetrieverResultItem(content="<Record n.name='Small Fry'>", metadata=None), RetrieverResultItem(content="<Record n.name='Small Fry'>", metadata=None), RetrieverResultItem(content="<Record n.name='Small Fry'>", metadata=None), RetrieverResultItem(content="<Record n.name='Small Fry'>", metadata=None), RetrieverResultItem(content="<Record n.name='Small Fry'>", metadata=None), RetrieverResultItem(content="<Record n.name='Small Fry'>", metadata=None), RetrieverResultItem(content="<Record n.name='Small Fry'>", metadata=None), RetrieverResultItem(content="<Record n.name='Glitters Team'>", metadata=None), RetrieverResultItem(content="<Record n.name='Glitters Team'>", metadata=None), RetrieverResultItem(content="<Record n.name='Glitters Team'>", metadata=None), RetrieverResultItem(content="<Record n.name='Glitters Team'>", metadata=None), RetrieverResultItem(content="<Record n.name='Glitters Team'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Oceanus City Council'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='Green Guardians'>", metadata=None), RetrieverResultItem(content="<Record n.name='V. Miesel Shipping'>", metadata=None), RetrieverResultItem(content="<Record n.name='V. Miesel Shipping'>", metadata=None), RetrieverResultItem(content="<Record n.name='V. Miesel Shipping'>", metadata=None), RetrieverResultItem(content="<Record n.name='V. Miesel Shipping'>", metadata=None), RetrieverResultItem(content="<Record n.name='V. Miesel Shipping'>", metadata=None), RetrieverResultItem(content="<Record n.name='V. Miesel Shipping'>", metadata=None), RetrieverResultItem(content="<Record n.name='V. Miesel Shipping'>", metadata=None), RetrieverResultItem(content="<Record n.name='V. Miesel Shipping'>", metadata=None), RetrieverResultItem(content="<Record n.name='V. Miesel Shipping'>", metadata=None), RetrieverResultItem(content="<Record n.name='V. Miesel Shipping'>", metadata=None), RetrieverResultItem(content="<Record n.name='V. Miesel Shipping'>", metadata=None), RetrieverResultItem(content="<Record n.name='V. Miesel Shipping'>", metadata=None), RetrieverResultItem(content="<Record n.name='V. Miesel Shipping'>", metadata=None), RetrieverResultItem(content="<Record n.name='V. Miesel Shipping'>", metadata=None), RetrieverResultItem(content="<Record n.name='V. Miesel Shipping'>", metadata=None), RetrieverResultItem(content="<Record n.name='V. Miesel Shipping'>", metadata=None), RetrieverResultItem(content="<Record n.name='V. Miesel Shipping'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sailor Shifts Team'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sailor Shifts Team'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sailor Shifts Team'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sailor Shifts Team'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sailor Shifts Team'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sailor Shifts Team'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sailor Shifts Team'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Neptune'>", metadata=None), RetrieverResultItem(content="<Record n.name='Marlin'>", metadata=None), RetrieverResultItem(content="<Record n.name='Marlin'>", metadata=None), RetrieverResultItem(content="<Record n.name='Marlin'>", metadata=None), RetrieverResultItem(content="<Record n.name='Marlin'>", metadata=None), RetrieverResultItem(content="<Record n.name='Marlin'>", metadata=None), RetrieverResultItem(content="<Record n.name='Marlin'>", metadata=None), RetrieverResultItem(content="<Record n.name='Marlin'>", metadata=None), RetrieverResultItem(content="<Record n.name='Marlin'>", metadata=None), RetrieverResultItem(content="<Record n.name='Serenity'>", metadata=None), RetrieverResultItem(content="<Record n.name='Serenity'>", metadata=None), RetrieverResultItem(content="<Record n.name='Serenity'>", metadata=None), RetrieverResultItem(content="<Record n.name='Serenity'>", metadata=None), RetrieverResultItem(content="<Record n.name='Serenity'>", metadata=None), RetrieverResultItem(content="<Record n.name='Serenity'>", metadata=None), RetrieverResultItem(content="<Record n.name='Serenity'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Mako'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Reef Guardian'>", metadata=None), RetrieverResultItem(content="<Record n.name='Horizon'>", metadata=None), RetrieverResultItem(content="<Record n.name='Horizon'>", metadata=None), RetrieverResultItem(content="<Record n.name='Horizon'>", metadata=None), RetrieverResultItem(content="<Record n.name='Horizon'>", metadata=None), RetrieverResultItem(content="<Record n.name='Horizon'>", metadata=None), RetrieverResultItem(content="<Record n.name='Horizon'>", metadata=None), RetrieverResultItem(content="<Record n.name='Horizon'>", metadata=None), RetrieverResultItem(content="<Record n.name='Horizon'>", metadata=None), RetrieverResultItem(content="<Record n.name='Horizon'>", metadata=None), RetrieverResultItem(content="<Record n.name='Horizon'>", metadata=None), RetrieverResultItem(content="<Record n.name='Horizon'>", metadata=None), RetrieverResultItem(content="<Record n.name='Horizon'>", metadata=None), RetrieverResultItem(content="<Record n.name='Horizon'>", metadata=None), RetrieverResultItem(content="<Record n.name='Horizon'>", metadata=None), RetrieverResultItem(content="<Record n.name='Horizon'>", metadata=None), RetrieverResultItem(content="<Record n.name='Horizon'>", metadata=None), RetrieverResultItem(content="<Record n.name='Horizon'>", metadata=None), RetrieverResultItem(content="<Record n.name='Horizon'>", metadata=None), RetrieverResultItem(content="<Record n.name='Seawatch'>", metadata=None), RetrieverResultItem(content="<Record n.name='Seawatch'>", metadata=None), RetrieverResultItem(content="<Record n.name='Seawatch'>", metadata=None), RetrieverResultItem(content="<Record n.name='Seawatch'>", metadata=None), RetrieverResultItem(content="<Record n.name='Seawatch'>", metadata=None), RetrieverResultItem(content="<Record n.name='Seawatch'>", metadata=None), RetrieverResultItem(content="<Record n.name='Seawatch'>", metadata=None), RetrieverResultItem(content="<Record n.name='Seawatch'>", metadata=None), RetrieverResultItem(content="<Record n.name='Seawatch'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='EcoVigil'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Sentinel'>", metadata=None), RetrieverResultItem(content="<Record n.name='Osprey'>", metadata=None), RetrieverResultItem(content="<Record n.name='Osprey'>", metadata=None), RetrieverResultItem(content="<Record n.name='Osprey'>", metadata=None), RetrieverResultItem(content="<Record n.name='Osprey'>", metadata=None), RetrieverResultItem(content="<Record n.name='Osprey'>", metadata=None), RetrieverResultItem(content="<Record n.name='Osprey'>", metadata=None), RetrieverResultItem(content="<Record n.name='Osprey'>", metadata=None), RetrieverResultItem(content="<Record n.name='Osprey'>", metadata=None), RetrieverResultItem(content="<Record n.name='Osprey'>", metadata=None), RetrieverResultItem(content="<Record n.name='Defender'>", metadata=None), RetrieverResultItem(content="<Record n.name='Defender'>", metadata=None), RetrieverResultItem(content="<Record n.name='Defender'>", metadata=None), RetrieverResultItem(content="<Record n.name='Defender'>", metadata=None), RetrieverResultItem(content="<Record n.name='Defender'>", metadata=None), RetrieverResultItem(content="<Record n.name='Defender'>", metadata=None), RetrieverResultItem(content="<Record n.name='Defender'>", metadata=None), RetrieverResultItem(content="<Record n.name='Northern Light'>", metadata=None), RetrieverResultItem(content="<Record n.name='Northern Light'>", metadata=None), RetrieverResultItem(content="<Record n.name='Northern Light'>", metadata=None), RetrieverResultItem(content="<Record n.name='Northern Light'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Remora'>", metadata=None), RetrieverResultItem(content="<Record n.name='Knowles'>", metadata=None), RetrieverResultItem(content="<Record n.name='Knowles'>", metadata=None), RetrieverResultItem(content="<Record n.name='Knowles'>", metadata=None), RetrieverResultItem(content="<Record n.name='Knowles'>", metadata=None), RetrieverResultItem(content="<Record n.name='Knowles'>", metadata=None), RetrieverResultItem(content="<Record n.name='Knowles'>", metadata=None), RetrieverResultItem(content="<Record n.name='Knowles'>", metadata=None), RetrieverResultItem(content="<Record n.name='Knowles'>", metadata=None), RetrieverResultItem(content='<Record n.name="Mariner\'s Dream">', metadata=None), RetrieverResultItem(content='<Record n.name="Mariner\'s Dream">', metadata=None), RetrieverResultItem(content='<Record n.name="Mariner\'s Dream">', metadata=None), RetrieverResultItem(content="<Record n.name='Recreational Fishing Boats'>", metadata=None), RetrieverResultItem(content="<Record n.name='Recreational Fishing Boats'>", metadata=None), RetrieverResultItem(content="<Record n.name='Recreational Fishing Boats'>", metadata=None), RetrieverResultItem(content="<Record n.name='Diving Tour Operators'>", metadata=None), RetrieverResultItem(content="<Record n.name='Tourists'>", metadata=None), RetrieverResultItem(content="<Record n.name='Tourists'>", metadata=None), RetrieverResultItem(content="<Record n.name='Conservation Vessels'>", metadata=None), RetrieverResultItem(content="<Record n.name='Conservation Vessels'>", metadata=None), RetrieverResultItem(content="<Record n.name='Conservation Vessels'>", metadata=None), RetrieverResultItem(content="<Record n.name='Conservation Vessels'>", metadata=None), RetrieverResultItem(content="<Record n.name='Paackland Harbor'>", metadata=None), RetrieverResultItem(content="<Record n.name='Paackland Harbor'>", metadata=None), RetrieverResultItem(content="<Record n.name='Paackland Harbor'>", metadata=None), RetrieverResultItem(content="<Record n.name='Paackland Harbor'>", metadata=None), RetrieverResultItem(content="<Record n.name='Paackland Harbor'>", metadata=None), RetrieverResultItem(content="<Record n.name='Paackland Harbor'>", metadata=None), RetrieverResultItem(content="<Record n.name='Haacklee Harbor'>", metadata=None), RetrieverResultItem(content="<Record n.name='Haacklee Harbor'>", metadata=None), RetrieverResultItem(content="<Record n.name='Himark Harbor'>", metadata=None), RetrieverResultItem(content="<Record n.name='Himark Harbor'>", metadata=None), RetrieverResultItem(content="<Record n.name='Himark Harbor'>", metadata=None), RetrieverResultItem(content="<Record n.name='Himark Harbor'>", metadata=None), RetrieverResultItem(content="<Record n.name='Port Security'>", metadata=None), RetrieverResultItem(content="<Record n.name='Port Security'>", metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content="<Record n.name='fish'>", metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None), RetrieverResultItem(content='<Record n.name=None>', metadata=None)] metadata={'cypher': 'cypher\nMATCH (n)-[:RELATED_TO]->(r)\nRETURN n.name\n', '__retriever': 'Text2CypherRetriever'}